# Example 4 (Python half): Prepare Per-Window GC Skew for a Statistical Test


Fetch the E. coli K-12 MG1655 genome, compute per-window GC skew, label each window by replichore , and save the table the R half of this Example will test statistically. **Use the Python kernel.**



In [ ]:
# Fetch the E. coli genome again and save per-window GC skew, labeled by replichore half
import csv
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com" # NCBI requires a real contact email
handle = Entrez.efetch(db="nucleotide", id="NC_000913.3", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()

genome_length = len(record.seq)

# Biological landmarks for E. coli K-12 MG1655 (from EcoCyc / published literature)
oriC = 3_923_767   # origin of replication, 84.57 min
terC = 1_607_000   # terC locus (terminus region)

# Non-overlapping 10,000 bp windows to ensure statistical independence
window = 10_000
step = 10_000      # step equals window to prevent pseudo-replication 

rows = []
cumulative = 0.0
for start in range(0, genome_length - window, step):
    chunk = record.seq[start:start + window]
    g, c = chunk.count("G"), chunk.count("C")
    if g + c == 0:                # avoid division by zero
        continue
    skew = (g - c) / (g + c)
    cumulative += skew            # cumulative GC skew

    # Label by replichore, not geometric midpoint
    # Replichore 1: positions < terC OR >= oriC
    # Replichore 2: positions >= terC AND < oriC
    if start < terC or start >= oriC:
        half = "replichore_1"
    else:
        half = "replichore_2"

    rows.append((round(start / 1_000_000, 3), round(skew, 4),
                 round(cumulative, 4), half))

with open("results/example4_skew_values.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["position_mb", "skew", "cumulative_skew", "replichore"])
    writer.writerows(rows)

print(f"Genome length: {genome_length:,} bp")
print(f"oriC at {oriC/1_000_000:.3f} Mb, terC at {terC/1_000_000:.3f} Mb")
print(f"Windows: {len(rows)} (non-overlapping)")
print("Saved: results/example4_skew_values.csv")

Genome length: 4,641,652 bp
oriC at 3.924 Mb, terC at 1.607 Mb
Windows: 464 (non-overlapping)
Saved: results/example4_skew_values.csv
